In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['figure.figsize'] = 12,6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 관련 ####################################################
# 결측치 처리
from sklearn.impute import SimpleImputer
# 표준화
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# 인코더
from sklearn.preprocessing import LabelEncoder
# 원핫 인코더
from sklearn.preprocessing import OneHotEncoder

# 학습 모델 성능 관련 ####################################################
# 원하는 비율로 데이터를 나누기 위해
from sklearn.model_selection import train_test_split
# K-Fold 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
# 학습곡선
from sklearn.model_selection import learning_curve
# 하이퍼 파라미터 튜닝
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import optuna
from optuna.samplers import TPESampler # 데이터를 랜덤샘플링하기 위함
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING) # 수치가 떨어지면 경고로그가 뜨는데 그거를 막아줌

# 모델 성능평가 #############################################
# 회귀용
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
# 분류용
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay

# 피처 선택 ################################################
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance

# 학습모델 ##################################################
#분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from catboost import CatBoostClassifier

#회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import BayesianRidge
from catboost import CatBoostRegressor

# 결정트리를 시각화할 수 있는 라이브러리
from sklearn.tree import plot_tree

# 차원축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE

# 연관규칙 학습
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.mixture import GaussianMixture
from sklearn.cluster import MeanShift, estimate_bandwidth

# 파이프라인
from sklearn.pipeline import Pipeline

# KDE를 그리기 위한 통계값을 구할 수 있는 함수
from scipy.stats import gaussian_kde

# 피어슨 상관 계수 (연속형 수치형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pearsonr
# 카이제곱 검증 (범주형 데이터 vs 범주현 데이터, 순위 x)
from scipy.stats import chi2_contingency
# 스피어만 상관계수 (범주형 데이터 vs 범주형 데이터, 순위 O)
from scipy.stats import spearmanr
# 포인트 이분 상관계수 (범주형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pointbiserialr

# 객체를 파일에 저장
import pickle

# 불필요한 경고 뜨지 않게
import warnings
warnings.filterwarnings('ignore')

### 데이터 불러오기

In [2]:
train_df = pd.read_csv('data/bike_sharing_train2.csv')
test_df = pd.read_csv('data/bike_sharing_test2.csv')

In [3]:
train_df

,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,log_count,year,month,day,hour,log_windspeed
0,1,0,0,1,9.84,14.395,81,0.0000,2.833213,2011,1,1,0,0.000000
1,1,0,0,1,9.02,13.635,80,0.0000,3.713572,2011,1,1,1,0.000000
2,1,0,0,1,9.02,13.635,80,0.0000,3.496508,2011,1,1,2,0.000000
3,1,0,0,1,9.84,14.395,75,0.0000,2.639057,2011,1,1,3,0.000000
4,1,0,0,1,9.84,14.395,75,0.0000,0.693147,2011,1,1,4,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,4,0,1,1,15.58,19.695,50,26.0027,5.820083,2012,12,19,19,3.295937
10882,4,0,1,1,14.76,17.425,57,15.0013,5.488938,2012,12,19,20,2.772670
10883,4,0,1,1,13.94,15.910,61,15.0013,5.129899,2012,12,19,21,2.772670
10884,4,0,1,1,13.94,17.425,61,6.0032,4.867534,2012,12,19,22,1.946367


### season

In [4]:
train_df['season'].value_counts()

season
4    2734
2    2733
3    2733
1    2686
Name: count, dtype: int64

In [5]:
# 값의 종류가 4가지 이므로 원핫 인코딩을 한다.
encoder = OneHotEncoder()
encoder.fit(train_df[['season']])

season_train_encoded = encoder.transform(train_df[['season']]).toarray()
season_test_encoded = encoder.transform(test_df[['season']]).toarray()

columns = encoder.get_feature_names_out(['season'])

encoded_train_df = pd.DataFrame(season_train_encoded, columns=columns)
encoded_test_df = pd.DataFrame(season_test_encoded, columns=columns)

train_df = pd.concat([train_df, encoded_train_df], axis=1)
test_df = pd.concat([test_df, encoded_test_df], axis=1)

train_df.drop('season',axis=1, inplace=True)
test_df.drop('season', axis=1, inplace=True)

display(train_df)
display(test_df)

,holiday,workingday,weather,temp,atemp,humidity,windspeed,log_count,year,month,day,hour,log_windspeed,season_1,season_2,season_3,season_4
0,0,0,1,9.84,14.395,81,0.0000,2.833213,2011,1,1,0,0.000000,1.0,0.0,0.0,0.0
1,0,0,1,9.02,13.635,80,0.0000,3.713572,2011,1,1,1,0.000000,1.0,0.0,0.0,0.0
2,0,0,1,9.02,13.635,80,0.0000,3.496508,2011,1,1,2,0.000000,1.0,0.0,0.0,0.0
3,0,0,1,9.84,14.395,75,0.0000,2.639057,2011,1,1,3,0.000000,1.0,0.0,0.0,0.0
4,0,0,1,9.84,14.395,75,0.0000,0.693147,2011,1,1,4,0.000000,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,0,1,1,15.58,19.695,50,26.0027,5.820083,2012,12,19,19,3.295937,0.0,0.0,0.0,1.0
10882,0,1,1,14.76,17.425,57,15.0013,5.488938,2012,12,19,20,2.772670,0.0,0.0,0.0,1.0
10883,0,1,1,13.94,15.910,61,15.0013,5.129899,2012,12,19,21,2.772670,0.0,0.0,0.0,1.0
10884,0,1,1,13.94,17.425,61,6.0032,4.867534,2012,12,19,22,1.946367,0.0,0.0,0.0,1.0


,holiday,workingday,weather,temp,atemp,humidity,windspeed,year,month,day,hour,log_windspeed,season_1,season_2,season_3,season_4
0,0,1,1,10.66,11.365,56,26.0027,2011,1,20,0,3.295937,1.0,0.0,0.0,0.0
1,0,1,1,10.66,13.635,56,0.0000,2011,1,20,1,0.000000,1.0,0.0,0.0,0.0
2,0,1,1,10.66,13.635,56,0.0000,2011,1,20,2,0.000000,1.0,0.0,0.0,0.0
3,0,1,1,10.66,12.880,56,11.0014,2011,1,20,3,2.485023,1.0,0.0,0.0,0.0
4,0,1,1,10.66,12.880,56,11.0014,2011,1,20,4,2.485023,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6488,0,1,2,10.66,12.880,60,11.0014,2012,12,31,19,2.485023,1.0,0.0,0.0,0.0
6489,0,1,2,10.66,12.880,60,11.0014,2012,12,31,20,2.485023,1.0,0.0,0.0,0.0
6490,0,1,1,10.66,12.880,60,11.0014,2012,12,31,21,2.485023,1.0,0.0,0.0,0.0
6491,0,1,1,10.66,13.635,56,8.9981,2012,12,31,22,2.302395,1.0,0.0,0.0,0.0


### holiday

In [6]:
train_df['holiday'].value_counts()

holiday
0    10575
1      311
Name: count, dtype: int64

### workingday

In [7]:
train_df['workingday'].value_counts()

workingday
1    7412
0    3474
Name: count, dtype: int64

### weather

In [8]:
train_df['weather'].value_counts()

weather
1    7192
2    2834
3     860
Name: count, dtype: int64

In [9]:
# 값 종류가 3가지이므로 원핫 인코딩
encoder = OneHotEncoder()
encoder.fit(train_df[['weather']])

weather_train_encoded = encoder.transform(train_df[['weather']]).toarray()
weather_test_encoded = encoder.transform(test_df[['weather']]).toarray()

columns = encoder.get_feature_names_out(['weather'])

encoded_train_df = pd.DataFrame(weather_train_encoded,columns=columns)
encoded_test_df = pd.DataFrame(weather_test_encoded, columns=columns)

train_df = pd.concat([train_df, encoded_train_df], axis=1)
test_df = pd.concat([test_df, encoded_test_df], axis=1)

train_df.drop('weather', axis=1,inplace=True)
test_df.drop('weather', axis=1, inplace=True)


display(train_df)
display(test_df)

,holiday,workingday,temp,atemp,humidity,windspeed,log_count,year,month,day,hour,log_windspeed,season_1,season_2,season_3,season_4,weather_1,weather_2,weather_3
0,0,0,9.84,14.395,81,0.0000,2.833213,2011,1,1,0,0.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0,0,9.02,13.635,80,0.0000,3.713572,2011,1,1,1,0.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0,0,9.02,13.635,80,0.0000,3.496508,2011,1,1,2,0.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0,0,9.84,14.395,75,0.0000,2.639057,2011,1,1,3,0.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0,0,9.84,14.395,75,0.0000,0.693147,2011,1,1,4,0.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,0,1,15.58,19.695,50,26.0027,5.820083,2012,12,19,19,3.295937,0.0,0.0,0.0,1.0,1.0,0.0,0.0
10882,0,1,14.76,17.425,57,15.0013,5.488938,2012,12,19,20,2.772670,0.0,0.0,0.0,1.0,1.0,0.0,0.0
10883,0,1,13.94,15.910,61,15.0013,5.129899,2012,12,19,21,2.772670,0.0,0.0,0.0,1.0,1.0,0.0,0.0
10884,0,1,13.94,17.425,61,6.0032,4.867534,2012,12,19,22,1.946367,0.0,0.0,0.0,1.0,1.0,0.0,0.0


,holiday,workingday,temp,atemp,humidity,windspeed,year,month,day,hour,log_windspeed,season_1,season_2,season_3,season_4,weather_1,weather_2,weather_3
0,0,1,10.66,11.365,56,26.0027,2011,1,20,0,3.295937,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0,1,10.66,13.635,56,0.0000,2011,1,20,1,0.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0,1,10.66,13.635,56,0.0000,2011,1,20,2,0.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0,1,10.66,12.880,56,11.0014,2011,1,20,3,2.485023,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0,1,10.66,12.880,56,11.0014,2011,1,20,4,2.485023,1.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6488,0,1,10.66,12.880,60,11.0014,2012,12,31,19,2.485023,1.0,0.0,0.0,0.0,0.0,1.0,0.0
6489,0,1,10.66,12.880,60,11.0014,2012,12,31,20,2.485023,1.0,0.0,0.0,0.0,0.0,1.0,0.0
6490,0,1,10.66,12.880,60,11.0014,2012,12,31,21,2.485023,1.0,0.0,0.0,0.0,1.0,0.0,0.0
6491,0,1,10.66,13.635,56,8.9981,2012,12,31,22,2.302395,1.0,0.0,0.0,0.0,1.0,0.0,0.0


### year

In [10]:
train_df['year'].value_counts()

year
2012    5464
2011    5422
Name: count, dtype: int64

In [11]:
# 2012 -> 0   /   2011 -> 1 로 변경
train_df['year'] = train_df['year'].map({
    2012 : 0,
    2011 : 1
})

test_df['year'] = test_df['year'].map({
    2012 : 0,
    2011 : 1
})

display(train_df['year'].value_counts())

year
0    5464
1    5422
Name: count, dtype: int64

### month

In [12]:
train_df['month'].value_counts()

month
5     912
6     912
7     912
8     912
12    912
10    911
11    911
4     909
9     909
2     901
3     901
1     884
Name: count, dtype: int64

In [13]:
# 12개 이므로 원핫인코딩 진행
encoder = OneHotEncoder()
encoder.fit(train_df[['month']])

month_train_encoded = encoder.transform(train_df[['month']]).toarray()
month_test_encoded = encoder.transform(test_df[['month']]).toarray()

columns = encoder.get_feature_names_out(['month'])

encoded_train_df = pd.DataFrame(month_train_encoded, columns=columns)
encoded_test_df = pd.DataFrame(month_test_encoded, columns=columns)

train_df = pd.concat([train_df, encoded_train_df], axis=1)
test_df = pd.concat([test_df,encoded_test_df], axis=1)

train_df.drop('month', axis=1, inplace=True)
test_df.drop('month', axis=1, inplace=True)

display(train_df)
display(test_df)

,holiday,workingday,temp,atemp,humidity,windspeed,log_count,year,day,hour,...,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,0,0,9.84,14.395,81,0.0000,2.833213,1,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,0,9.02,13.635,80,0.0000,3.713572,1,1,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,0,9.02,13.635,80,0.0000,3.496508,1,1,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,0,9.84,14.395,75,0.0000,2.639057,1,1,3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,0,9.84,14.395,75,0.0000,0.693147,1,1,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,0,1,15.58,19.695,50,26.0027,5.820083,0,19,19,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
10882,0,1,14.76,17.425,57,15.0013,5.488938,0,19,20,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
10883,0,1,13.94,15.910,61,15.0013,5.129899,0,19,21,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
10884,0,1,13.94,17.425,61,6.0032,4.867534,0,19,22,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


,holiday,workingday,temp,atemp,humidity,windspeed,year,day,hour,log_windspeed,...,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,0,1,10.66,11.365,56,26.0027,1,20,0,3.295937,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,1,10.66,13.635,56,0.0000,1,20,1,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,1,10.66,13.635,56,0.0000,1,20,2,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,1,10.66,12.880,56,11.0014,1,20,3,2.485023,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,1,10.66,12.880,56,11.0014,1,20,4,2.485023,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6488,0,1,10.66,12.880,60,11.0014,0,31,19,2.485023,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
6489,0,1,10.66,12.880,60,11.0014,0,31,20,2.485023,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
6490,0,1,10.66,12.880,60,11.0014,0,31,21,2.485023,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
6491,0,1,10.66,13.635,56,8.9981,0,31,22,2.302395,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


### day

In [14]:
train_df['day'].value_counts()

day
1     575
5     575
9     575
17    575
4     574
7     574
8     574
13    574
14    574
15    574
16    574
19    574
2     573
3     573
12    573
6     572
10    572
11    568
18    563
Name: count, dtype: int64

In [16]:
test_df['day'].value_counts()

day
20    574
21    574
23    573
24    573
25    572
22    569
26    567
28    563
27    552
29    526
30    514
31    336
Name: count, dtype: int64

In [17]:
# train_df 와 test_df 간의 날짜 데이터는 서로 다르게 되어있다.
# 이에 해당 컬럼은 제거한다.
train_df.drop('day', axis=1, inplace=True)
test_df.drop('day', axis=1, inplace=True)

display(train_df)
display(test_df)

,holiday,workingday,temp,atemp,humidity,windspeed,log_count,year,hour,log_windspeed,...,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,0,0,9.84,14.395,81,0.0000,2.833213,1,0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,0,9.02,13.635,80,0.0000,3.713572,1,1,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,0,9.02,13.635,80,0.0000,3.496508,1,2,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,0,9.84,14.395,75,0.0000,2.639057,1,3,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,0,9.84,14.395,75,0.0000,0.693147,1,4,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,0,1,15.58,19.695,50,26.0027,5.820083,0,19,3.295937,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
10882,0,1,14.76,17.425,57,15.0013,5.488938,0,20,2.772670,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
10883,0,1,13.94,15.910,61,15.0013,5.129899,0,21,2.772670,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
10884,0,1,13.94,17.425,61,6.0032,4.867534,0,22,1.946367,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


,holiday,workingday,temp,atemp,humidity,windspeed,year,hour,log_windspeed,season_1,...,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,0,1,10.66,11.365,56,26.0027,1,0,3.295937,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,1,10.66,13.635,56,0.0000,1,1,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,1,10.66,13.635,56,0.0000,1,2,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,1,10.66,12.880,56,11.0014,1,3,2.485023,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,1,10.66,12.880,56,11.0014,1,4,2.485023,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6488,0,1,10.66,12.880,60,11.0014,0,19,2.485023,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
6489,0,1,10.66,12.880,60,11.0014,0,20,2.485023,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
6490,0,1,10.66,12.880,60,11.0014,0,21,2.485023,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
6491,0,1,10.66,13.635,56,8.9981,0,22,2.302395,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


### hour

In [18]:
train_df['hour'].value_counts()

hour
12    456
13    456
14    456
15    456
16    456
17    456
18    456
19    456
20    456
21    456
22    456
23    456
0     455
6     455
7     455
8     455
9     455
10    455
11    455
1     454
5     452
2     448
4     442
3     433
Name: count, dtype: int64

In [20]:
encoder = OneHotEncoder()
encoder.fit(train_df[['hour']])

hour_train_encoded = encoder.transform(train_df[['hour']]).toarray()
hour_test_encoded = encoder.transform(test_df[['hour']]).toarray()

columns = encoder.get_feature_names_out(['hour'])

encoded_train_df = pd.DataFrame(hour_train_encoded, columns=columns)
encoded_test_df = pd.DataFrame(hour_test_encoded, columns=columns)

train_df = pd.concat([train_df,encoded_train_df], axis=1)
test_df = pd.concat([test_df,encoded_test_df], axis=1)

train_df.drop('hour',axis=1,inplace=True)
test_df.drop('hour', axis=1,inplace=True)

display(train_df)
display(test_df)

,holiday,workingday,temp,atemp,humidity,windspeed,log_count,year,log_windspeed,season_1,...,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,0,0,9.84,14.395,81,0.0000,2.833213,1,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,0,9.02,13.635,80,0.0000,3.713572,1,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,0,9.02,13.635,80,0.0000,3.496508,1,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,0,9.84,14.395,75,0.0000,2.639057,1,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,0,9.84,14.395,75,0.0000,0.693147,1,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,0,1,15.58,19.695,50,26.0027,5.820083,0,3.295937,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
10882,0,1,14.76,17.425,57,15.0013,5.488938,0,2.772670,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
10883,0,1,13.94,15.910,61,15.0013,5.129899,0,2.772670,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
10884,0,1,13.94,17.425,61,6.0032,4.867534,0,1.946367,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


,holiday,workingday,temp,atemp,humidity,windspeed,year,log_windspeed,season_1,season_2,...,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,0,1,10.66,11.365,56,26.0027,1,3.295937,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,1,10.66,13.635,56,0.0000,1,0.000000,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,1,10.66,13.635,56,0.0000,1,0.000000,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,1,10.66,12.880,56,11.0014,1,2.485023,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,1,10.66,12.880,56,11.0014,1,2.485023,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6488,0,1,10.66,12.880,60,11.0014,0,2.485023,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
6489,0,1,10.66,12.880,60,11.0014,0,2.485023,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
6490,0,1,10.66,12.880,60,11.0014,0,2.485023,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6491,0,1,10.66,13.635,56,8.9981,0,2.302395,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [21]:
train_df.isna().sum()

holiday          0
workingday       0
temp             0
atemp            0
humidity         0
windspeed        0
log_count        0
year             0
log_windspeed    0
season_1         0
season_2         0
season_3         0
season_4         0
weather_1        0
weather_2        0
weather_3        0
month_1          0
month_2          0
month_3          0
month_4          0
month_5          0
month_6          0
month_7          0
month_8          0
month_9          0
month_10         0
month_11         0
month_12         0
hour_0           0
hour_1           0
hour_2           0
hour_3           0
hour_4           0
hour_5           0
hour_6           0
hour_7           0
hour_8           0
hour_9           0
hour_10          0
hour_11          0
hour_12          0
hour_13          0
hour_14          0
hour_15          0
hour_16          0
hour_17          0
hour_18          0
hour_19          0
hour_20          0
hour_21          0
hour_22          0
hour_23          0
dtype: int64

In [22]:
test_df.isna().sum()

holiday          0
workingday       0
temp             0
atemp            0
humidity         0
windspeed        0
year             0
log_windspeed    0
season_1         0
season_2         0
season_3         0
season_4         0
weather_1        0
weather_2        0
weather_3        0
month_1          0
month_2          0
month_3          0
month_4          0
month_5          0
month_6          0
month_7          0
month_8          0
month_9          0
month_10         0
month_11         0
month_12         0
hour_0           0
hour_1           0
hour_2           0
hour_3           0
hour_4           0
hour_5           0
hour_6           0
hour_7           0
hour_8           0
hour_9           0
hour_10          0
hour_11          0
hour_12          0
hour_13          0
hour_14          0
hour_15          0
hour_16          0
hour_17          0
hour_18          0
hour_19          0
hour_20          0
hour_21          0
hour_22          0
hour_23          0
dtype: int64

In [23]:
train_df.to_csv('data/bike_sharing_train3.csv')
test_df.to_csv('data/bike_sharing_test3.csv')